# Generate a synthetic card dataset

Stage one predicts card boxes (COCO category: card). Use the grouped controls to select either reference deck and narrow appearance ranges. Card height is derived from width and a width/height ratio of 0.64–0.69. All shaded patterns are selected by default.


In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from set_master.generation.synthetic_dataset import DatasetConfig, generate_coco_dataset
from set_master.models import Color, Shading, ShadingPattern, Shape
from set_master.paths import SYNTHETIC_DATASETS_DIR

defaults = DatasetConfig()
patterns = widgets.SelectMultiple(options=[(item.value.replace('_', ' ').title(), item) for item in ShadingPattern], value=defaults.shading_patterns, description='Patterns')
shapes = widgets.SelectMultiple(options=[(item.name.title(), item) for item in Shape], value=defaults.shapes, description='Shapes')
colors = widgets.SelectMultiple(options=[(item.name.title(), item) for item in Color], value=defaults.colors, description='Colors')
shadings = widgets.SelectMultiple(options=[(item.value.title(), item) for item in Shading], value=defaults.shadings, description='Shadings')
image_count = widgets.IntText(value=defaults.image_count, description='Images')
image_width = widgets.IntText(value=defaults.image_width, description='Image width')
image_height = widgets.IntText(value=defaults.image_height, description='Image height')
cards_per_image = widgets.IntRangeSlider(value=defaults.cards_per_image, min=0, max=20, description='Cards/image', continuous_update=False)
rotation_degrees = widgets.FloatRangeSlider(value=defaults.rotation_degrees, min=-180, max=180, step=1, description='Rotation', continuous_update=False)
skew_x_degrees = widgets.FloatRangeSlider(value=defaults.skew_x_degrees, min=-30, max=30, step=1, description='Skew X', continuous_update=False)
skew_y_degrees = widgets.FloatRangeSlider(value=defaults.skew_y_degrees, min=-30, max=30, step=1, description='Skew Y', continuous_update=False)
perspective_jitter = widgets.FloatRangeSlider(value=defaults.perspective_jitter, min=0, max=0.3, step=0.01, description='Perspective', continuous_update=False)
background_gray = widgets.IntSlider(value=defaults.background_gray, min=0, max=255, description='Background', continuous_update=False)
seed = widgets.IntText(value=defaults.seed, description='Seed')
output_root = widgets.Text(value=str(SYNTHETIC_DATASETS_DIR), description='Output folder')

# Keep appearance ranges within the modest variation observed in the reference decks.
range_widgets = {}
for name in ('card_width', 'card_aspect_ratio', 'feature_count', 'feature_width_ratio', 'feature_height_ratio', 'symbol_spacing_ratio', 'corner_radius_ratio', 'stripe_spacing', 'line_thickness', 'squiggle_amplitude', 'squiggle_waves', 'squiggle_thickness'):
    low, high = getattr(defaults, name)
    slider_type = widgets.IntRangeSlider if isinstance(low, int) else widgets.FloatRangeSlider
    range_widgets[name] = slider_type(
        value=(low, high), min=low, max=high, step=1 if isinstance(low, int) else 0.001,
        description=name.replace('_', ' '), continuous_update=False,
        style={'description_width': '180px'}, layout=widgets.Layout(width='500px'),
    )
scene_group = widgets.VBox([image_count, image_width, image_height, cards_per_image, rotation_degrees, skew_x_degrees, skew_y_degrees, perspective_jitter, background_gray, seed, output_root])
card_group = widgets.VBox([shapes, colors, shadings] + [range_widgets[name] for name in ('card_width', 'card_aspect_ratio', 'feature_count', 'feature_width_ratio', 'feature_height_ratio', 'symbol_spacing_ratio', 'corner_radius_ratio')])
pattern_group = widgets.VBox([patterns, range_widgets['stripe_spacing'], range_widgets['line_thickness']])
squiggle_group = widgets.VBox([range_widgets[name] for name in ('squiggle_amplitude', 'squiggle_waves', 'squiggle_thickness')])
groups = widgets.Accordion(children=[scene_group, card_group, pattern_group, squiggle_group])
for index, title in enumerate(['Scene and output', 'Card and symbols', 'Shaded patterns', 'Squiggle geometry']):
    groups.set_title(index, title)
generate_button = widgets.Button(description='Generate dataset', button_style='primary')
status = widgets.Output()

def generate(_):
    with status:
        status.clear_output()
        try:
            config = DatasetConfig(
                output_root=Path(output_root.value), image_count=image_count.value,
                image_width=image_width.value, image_height=image_height.value,
                cards_per_image=tuple(cards_per_image.value), rotation_degrees=tuple(rotation_degrees.value),
                perspective_jitter=tuple(perspective_jitter.value),
                skew_x_degrees=tuple(skew_x_degrees.value), skew_y_degrees=tuple(skew_y_degrees.value),
                background_gray=background_gray.value, seed=seed.value,
                shading_patterns=tuple(patterns.value), shapes=tuple(shapes.value),
                colors=tuple(colors.value), shadings=tuple(shadings.value),
                **{name: tuple(slider.value) for name, slider in range_widgets.items()},
            )
        except ValueError as error:
            print(error)
            return
        print(f'Dataset saved to {generate_coco_dataset(config)}')

generate_button.on_click(generate)
display(widgets.VBox([groups, generate_button, status]))
